In [1]:
!sudo apt-get update

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done


In [2]:
!sudo apt-get update && sudo apt-get install -y build-essential

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
0 upgraded, 0 newly installed, 0 to remove and 59 not upgraded.


In [3]:
!sudo apt-get install -y python3.10-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libexpat1 libexpat1-dev libpython3.10 libpython3.10-dev
  libpython3.10-minimal libpython3.10-stdlib python3.10 python3.10-minimal
  zlib1g-dev
Suggested packages:
  python3.10-venv python3.10-doc binfmt-support
The following NEW packages will be installed:
  libexpat1-dev libpython3.10 libpython3.10-dev python3.10-dev zlib1g-dev
The following packages will be upgraded:
  libexpat1 libpython3.10-minimal libpython3.10-stdlib python3.10
  python3.10-minimal
5 upgraded, 5 newly installed, 0 to remove and 54 not upgraded.
Need to get 13.1 MB of archives.
After this operation, 29.0 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3.10 amd64 3.10.12-1~22.04.12 [508 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpython3.10-stdlib amd64 3.10.12-1~22.04.12 [1,

In [4]:
%%capture
import os
import sys


!{sys.executable} -m pip install pip3-autoremove
!{sys.executable} -m pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!{sys.executable} -m pip install unsloth
!{sys.executable} -m pip install transformers==4.56.2
!{sys.executable} -m pip install --no-deps trl==0.22.2

In [5]:
import sys
!{sys.executable} -m pip install nltk rouge_score absl-py
!{sys.executable} -m pip install bert_score
!{sys.executable} -m pip install scikit-learn


Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py) ... done
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 124.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=9487391cb089a944a1fb25a18e208ebdf9f2545f0d44455daf834f53ce937a18
  Stored in directory: /home/admin/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 187.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA H200


In [3]:
from huggingface_hub import login

login()

In [4]:
from unsloth import FastLanguageModel
import torch


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    dtype = None,
    load_in_4bit=True,
    token=hf_token
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.5: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H200. Num GPUs = 1. Max memory: 139.719 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.12.5 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [6]:
from datasets import load_dataset
dataset = load_dataset("lqkhoi/viet_med_qa", split = "train")


In [7]:
data = dataset.train_test_split(test_size=0.1, shuffle=True, seed = 42)


In [8]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

In [9]:
train_data = data['train']
test_data = data['test']

In [10]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

In [11]:
train_data[0]

{'url': 'https://www.vinmec.com//vie/bai-viet/met-moi-co-phai-la-benh-vi',
 'tags': ['Lối sống lành mạnh',
  'Hội chứng mệt mỏi',
  'Trầm cảm',
  'Dinh dưỡng khoa học',
  'Suy nhược cơ thể',
  'Bệnh lý thần kinh'],
 'question': 'Điều trị hội chứng mệt mỏi mạn tính như thế nào?',
 'answer': 'Hiện tại, không có biện pháp đặc trị. Điều trị chỉ nhằm mục đích làm giảm các triệu chứng. Nhìn chung, những bệnh nhân mắc hội chứng mệt mỏi được chẩn đoán trong 2 năm đầu kể từ khi xuất hiện các triệu chứng sẽ đáp ứng với điều trị tốt hơn. Phương pháp điều trị để giảm các triệu chứng phải được cá nhân hóa cho từng bệnh nhân. Chế độ ăn uống khoa học, bổ sung dinh dưỡng đầy đủ, sinh hoạt lành mạnh, làm việc và nghỉ ngơi hợp lý là vấn đề cốt lõi của bất kỳ phương pháp điều trị nào. Đây cũng là biện pháp hiệu quả nhất để ngăn ngừa hội chứng mệt mỏi xuất hiện trong cuộc sống hiện đại.'}

In [12]:
test_data[0]

{'url': 'https://www.vinmec.com//vie/bai-viet/cac-roi-loan-huyet-dong-dac-trung-trong-soc-tim-vi',
 'tags': ['Tim mạch',
  'Bị sốc tim',
  'Sốc tim',
  'Rối loạn huyết động',
  'Huyết động học'],
 'question': 'Nêu các dấu hiệu nhận biết tình trạng sốc tim?',
 'answer': 'Sốc tim là tình trạng tim đột nhiên không thể bơm máu nhằm đáp ứng nhu cầu cơ thể. Bị sốc tim thường hiếm, nhưng nếu không điều trị kịp thời thì người bệnh có thể bị đe dọa đến tính mạng, thậm chí tử vong. Các dấu hiệu có thể xảy ra khi bị sốc tim gồm:Vã mồ hôi, da nhợt nhạt hoặc xanh tái, chi lạnhÝ thức mất, huyết áp hạ hoặc không đo đượcNgười bệnh thở nhanh kèm khó thở, nhịp tim nhanhTiểu ít hoặc bí tiểuĐa phần các trường hợp tim bị thiếu oxy là do cơn đau thắt ngực và khi máu giàu oxy không lưu thông được đến khu vực buồng tim thì cơ tim yếu và gây sốc tim. Bên cạnh đó, các nguyên nhân khác có thể gây sốc tim là:Viêm cơ tim hoặc nhiễm trùng van timSuy timSử dụng thuốc quá liều ảnh hưởng đến khả năng bơm máu của timLo

In [13]:
EOS_TOKEN = tokenizer.eos_token
def format_with_system_prompt(example):
  example['conversations'] = [
        {"role": "user", "content": example['question']},
        {"role": "assistant", "content": example['answer']}
  ]
  return example

train_data = train_data.map(format_with_system_prompt)
test_data = test_data.map(format_with_system_prompt)


def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
   return { "text" : texts, }

train_data = train_data.map(formatting_prompts_func, batched = True)
test_data = test_data.map(formatting_prompts_func, batched = True)

In [14]:
train_data[0]

{'url': 'https://www.vinmec.com//vie/bai-viet/met-moi-co-phai-la-benh-vi',
 'tags': ['Lối sống lành mạnh',
  'Hội chứng mệt mỏi',
  'Trầm cảm',
  'Dinh dưỡng khoa học',
  'Suy nhược cơ thể',
  'Bệnh lý thần kinh'],
 'question': 'Điều trị hội chứng mệt mỏi mạn tính như thế nào?',
 'answer': 'Hiện tại, không có biện pháp đặc trị. Điều trị chỉ nhằm mục đích làm giảm các triệu chứng. Nhìn chung, những bệnh nhân mắc hội chứng mệt mỏi được chẩn đoán trong 2 năm đầu kể từ khi xuất hiện các triệu chứng sẽ đáp ứng với điều trị tốt hơn. Phương pháp điều trị để giảm các triệu chứng phải được cá nhân hóa cho từng bệnh nhân. Chế độ ăn uống khoa học, bổ sung dinh dưỡng đầy đủ, sinh hoạt lành mạnh, làm việc và nghỉ ngơi hợp lý là vấn đề cốt lõi của bất kỳ phương pháp điều trị nào. Đây cũng là biện pháp hiệu quả nhất để ngăn ngừa hội chứng mệt mỏi xuất hiện trong cuộc sống hiện đại.',
 'conversations': [{'content': 'Điều trị hội chứng mệt mỏi mạn tính như thế nào?',
   'role': 'user'},
  {'content': '

In [15]:
test_data[0]

{'url': 'https://www.vinmec.com//vie/bai-viet/cac-roi-loan-huyet-dong-dac-trung-trong-soc-tim-vi',
 'tags': ['Tim mạch',
  'Bị sốc tim',
  'Sốc tim',
  'Rối loạn huyết động',
  'Huyết động học'],
 'question': 'Nêu các dấu hiệu nhận biết tình trạng sốc tim?',
 'answer': 'Sốc tim là tình trạng tim đột nhiên không thể bơm máu nhằm đáp ứng nhu cầu cơ thể. Bị sốc tim thường hiếm, nhưng nếu không điều trị kịp thời thì người bệnh có thể bị đe dọa đến tính mạng, thậm chí tử vong. Các dấu hiệu có thể xảy ra khi bị sốc tim gồm:Vã mồ hôi, da nhợt nhạt hoặc xanh tái, chi lạnhÝ thức mất, huyết áp hạ hoặc không đo đượcNgười bệnh thở nhanh kèm khó thở, nhịp tim nhanhTiểu ít hoặc bí tiểuĐa phần các trường hợp tim bị thiếu oxy là do cơn đau thắt ngực và khi máu giàu oxy không lưu thông được đến khu vực buồng tim thì cơ tim yếu và gây sốc tim. Bên cạnh đó, các nguyên nhân khác có thể gây sốc tim là:Viêm cơ tim hoặc nhiễm trùng van timSuy timSử dụng thuốc quá liều ảnh hưởng đến khả năng bơm máu của timLo

In [17]:
!pip install wandb

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 84.5 MB/s eta 0:00:00ta 0:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [16]:
wandb login


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /home/admin/.netrc
wandb: Currently logged in as: vietanhm6a (vietanhm6a-hanoi-university-of-science-and-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [18]:
import os
os.environ["WANDB_PROJECT"] = "Medical-Chatbot" 
os.environ["WANDB_LOG_MODEL"] = "false"             

In [20]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    eval_dataset = test_data,
    dataset_text_field = "text",
    max_seq_length = 2048,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 4, # Set this for 1 full training run.
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/9590 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/1066 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [21]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map (num_proc=64):   0%|          | 0/9590 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/1066 [00:00<?, ? examples/s]

In [22]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                       Leptospira là bệnh truyền nhiễm cấp tính, do xoắn khuẩn Leptospira gây ra. Bệnh có thể gặp ở nhiều nước trên thế giới. Bệnh lây truyền từ động vật sang người, qua đường da và niêm mạc. Bệnh do Leptospira gây ra có đặc trưng bởi hội chứng nhiễm khuẩn nhiễm độc toàn thân, hội chứng gan thận.<|im_end|>\n'

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,590 | Num Epochs = 4 | Total steps = 4,796
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.006600
2,2.705900
3,2.534000
4,2.238900
5,2.118300
6,2.234900
7,1.975900
8,2.104300
9,1.931300
10,1.895600


In [24]:
messages = [
    {"role" : "user", "content" : "Điều trị hội chứng mệt mỏi mạn tính như thế nào?"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 4096, # Increase for longer outputs!
    # temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Hiện nay chưa có biện pháp điều trị đặc hiệu và cụ thể cho hội chứng mệt mỏi mạn tính. Việc điều trị chủ yếu là điều trị triệu chứng và hỗ trợ điều trị. Cụ thể là:Điều trị các bệnh đi kèm;Sử dụng thuốc giảm đau, thuốc chống trầm cảm, thuốc chống co giật;Duy trì chế độ ăn uống lành mạnh, bổ sung dinh dưỡng đầy đủ;Thường xuyên tập luyện thể dục thể thao;Thực hiện các liệu pháp tâm lý;Liệu pháp tập luyện tim mạch;Liệu pháp tập luyện thần kinh cơ;Liệu pháp vận động;Liệu pháp quang điện;Liệu pháp nhiệt.<|im_end|>


In [25]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
    token=hf_token
)

==((====))==  Unsloth 2025.12.5: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H200. Num GPUs = 1. Max memory: 139.719 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [26]:
FastLanguageModel.for_inference(model)
FastLanguageModel.for_inference(base_model)


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
    (layers): ModuleList(
      (0): Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear4bit(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Q

In [29]:
def infer_single(model, tokenizer, question):
    messages = [
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    output_ids = model.generate(
        **tokenizer(text, return_tensors="pt").to("cuda"),
        max_new_tokens=2048,
    )
    
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return generated.split("assistant")[-1].strip()


In [32]:
_ = infer_single(model, tokenizer, train_data[1]['question'])

In [33]:
print(_)

Viêm túi mật cấp là tình trạng viêm cấp tính của túi mật do các nguyên nhân khác nhau. Các nguyên nhân gây viêm túi mật cấp gồm có: Nang sỏi túi mật: Đây là nguyên nhân thường gặp nhất gây viêm túi mật cấp. Khi có sỏi trong túi mật, các cơn co thắt hoặc sỏi rơi vào ống mật chủ sẽ làm tắc nghẽn dòng chảy của mật, gây nên tình trạng viêm cấp túi mật. Nang túi mật: Người bị viêm túi mật có mảng lắng đọng trong niêm mạc túi mật, lắng đọng lipid trong niêm mạc túi mật, viêm túi mật mạn tính, viêm túi mật do giun chui, giun chỉ, sỏi túi mật, viêm đường mật cấp hoặc mạn tính, viêm túi mật do áp xe gan, nang gan, tắc mật do u đường mật, viêm túi mật do tụy, viêm túi mật do dị dạng bẩm sinh... đều có nguy cơ gây viêm cấp túi mật.


In [ ]:
test_data = data['test']

In [ ]:
test_data[0]

In [34]:
predictions_base = []
predictions_ft   = []
i=0
for item in test_data:
    instr = item["question"]
    inp   = ""
    gold  = item["answer"]

    pred_base = infer_single(base_model, base_tokenizer, instr)
    pred_ft   = infer_single(model, tokenizer, instr)
    predictions_base.append((pred_base, gold))
    predictions_ft.append((pred_ft, gold))
    print(i)
    i = i + 1
    if i == 100: break


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99


In [35]:
model.push_to_hub("NgVietAnh41/Qwen3-4b-it-test-VietMedQA-qlora", token=hf_token)
tokenizer.push_to_hub("NgVietAnh41/Qwen3-4b-it-test-VietMedQA-qlora", token=hf_token)

README.md:   0%|          | 0.00/618 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/NgVietAnh41/Qwen3-4b-it-test-VietMedQA-qlora


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [ ]:
test_data = data['test']
test_data.to_json("test.jsonl", lines=True) 

In [37]:
model.push_to_hub_gguf("NgVietAnh41/Qwen3-4b-it-test-VietMedQA-gguf", tokenizer, quantization_method = "q4_k_m", token = hf_token)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /home/admin/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen3-4b-instruct-2507.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['qwen3-4b-instruct-2507.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: llama-cli --model qwen3-4b-instruct-2507.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to current directory
Unsloth: convert model to ollama format by running - ollama create model_name -f ./Modelfile - inside current directory.
Unsloth: Uploading GGUF to Huggingface Hub...
Uploading qwen3-4b-instruct-2507.Q4_K_M.gguf

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/NgVietAnh41/Qwen3-4b-it-test-VietMedQA-gguf
Unsloth: Cleaning up temporary files...


'NgVietAnh41/Qwen3-4b-it-test-VietMedQA-gguf'

In [36]:
model.push_to_hub_merged("NgVietAnh41/Qwen3-4b-it-test-VietMedQA", tokenizer, save_method = "merged_16bit", token = hf_token)


config.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /home/admin/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [00:46<00:46, 46.81s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:23<00:00, 41.66s/it]


Unsloth: Merge process complete. Saved to `/home/admin/NgVietAnh41/Qwen3-4b-it-test-VietMedQA`


In [40]:
import sys
!{sys.executable} -m pip install evaluate


Defaulting to user installation because normal site-packages is not writeable
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [41]:
from evaluate import load
bleu      = load("bleu")
rouge     = load("rouge")
bertscore = load("bertscore")
f1_metric = load("f1")
squad_metric = load("squad")

import numpy as np


def exact_match(pred, ref):
    return int(pred.strip() == ref.strip())

def f1_score(pred, ref):
    return f1_metric.compute(predictions=[pred], references=[ref])["f1"]


# Compute metrics
def compute_metrics(predictions):
    preds  = [p[0] for p in predictions]
    golds  = [p[1] for p in predictions]

    # --- format cho squad_metric ---
    squad_predictions = [
        {"id": str(i), "prediction_text": preds[i]}
        for i in range(len(preds))
    ]

    squad_references = [
        {"id": str(i), "answers": {"text": [golds[i]], "answer_start": [0]}}
        for i in range(len(golds))
    ]

    return {
        "BLEU": bleu.compute(predictions=preds, references=[[g] for g in golds])["bleu"],
        "ROUGE-L": rouge.compute(predictions=preds, references=golds)["rougeL"],
        "Exact_Match": sum(exact_match(p, g) for p, g in predictions) / len(predictions),
        "BERTScore": np.mean(bertscore.compute(predictions=preds, references=golds, lang="en")["f1"]),
        "F1+EM": squad_metric.compute(
            predictions=squad_predictions,
            references=squad_references
        )
    }


metrics_base = compute_metrics(predictions_base)
metrics_ft   = compute_metrics(predictions_ft)

# Print results
print("=== Base Model ===")
print(metrics_base)
print("=== Fine-tuned Model ===")
print(metrics_ft)


# Save comparison to JSON
import json
with open("eval_comparison1.json", "w", encoding="utf-8") as f:
    json.dump({"base": metrics_base, "fine_tuned": metrics_ft}, f, indent=2)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


=== Base Model ===
{'BLEU': 0.028719131619127255, 'ROUGE-L': np.float64(0.22780177228503534), 'Exact_Match': 0.0, 'BERTScore': np.float64(0.8858197575807571), 'F1+EM': {'exact_match': 0.0, 'f1': 23.75537866139124}}
=== Fine-tuned Model ===
{'BLEU': 0.1003852617510709, 'ROUGE-L': np.float64(0.3349845956764994), 'Exact_Match': 0.0, 'BERTScore': np.float64(0.8964263153076172), 'F1+EM': {'exact_match': 0.0, 'f1': 31.918780666400085}}
